In [ ]:
!pip install transformers torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct") 

In [ ]:
!pip install peft # Parameter Efficient Fine Tuning
from peft import LoraConfig, get_peft_model, TaskType
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, config)
model.print_trainable_parameters()  # a lot smaller than the full model, this is the advantage of lora

# r is the size of the matrices we create with LoRA. it creates two matrixes of size (d_model, r) and (r, d_model) where d_model is the size of the model's hidden states.
# alpha is a scaling factor (a/r)

In [ ]:
def chat(model, tokenizer, user_message):  
    messages = [{"role": "user", "content": user_message}]

    # see the tokens
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    print(input_ids)
    # see the raw output 
    output = model.generate(**input_ids, max_new_tokens=100, pad_token_id=tokenizer.eos_token_id) 
    print(output)

    # see the response (slice off input tokens)
    response = tokenizer.decode(output[0][input_ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

In [ ]:
print(chat(model, tokenizer, "hi, how are you?"))